In [ ]:
/content/drive/MyDrive/대학부 데이터

In [ ]:
# Colab에서 Google Drive를 사용할 수 있도록 도구를 가져옵니다.
from google.colab import drive

# Google Drive를 Colab에 연결합니다.
drive.mount("/content/drive")

In [ ]:
# 파일과 폴더의 위치를 편리하게 처리하는 도구입니다.
from pathlib import Path


# 내 드라이브에 추가한 '대학부 데이터' 폴더의 위치입니다.
DATA_DIR = Path(
    "/content/drive/MyDrive/대학부 데이터"
)


# 모델 학습에 사용하는 서울 Training JSON 폴더입니다.
TRAIN_LABEL_DIR = (
    DATA_DIR
    / "Training"
    / "2.라벨링데이터"
    / "TL_서울_구급"
)


# 최종 추론에 사용하는 서울 Validation JSON 폴더입니다.
VALID_LABEL_DIR = (
    DATA_DIR
    / "Validation"
    / "2.라벨링데이터"
    / "VL_서울_구급"
)


# 실행 규칙에서 요구하는 Validation WAV 폴더입니다.
# 미션 3 모델은 음성을 실제 입력으로 사용하지 않습니다.
VALID_AUDIO_DIR = (
    DATA_DIR
    / "Validation"
    / "1.원천데이터"
    / "VS_서울_구급"
)


# 각 폴더가 정상적으로 연결되었는지 확인합니다.
print("Training JSON 폴더:", TRAIN_LABEL_DIR.exists())
print("Validation JSON 폴더:", VALID_LABEL_DIR.exists())
print("Validation WAV 폴더:", VALID_AUDIO_DIR.exists())

In [ ]:
# Training 폴더 안의 모든 JSON 파일을 찾습니다.
train_json_files = sorted(
    TRAIN_LABEL_DIR.glob("*.json")
)


# 발견한 JSON 파일 개수를 출력합니다.
print(
    "Training JSON 개수:",
    len(train_json_files),
)


# 처음 발견한 JSON 파일 5개를 출력합니다.
for json_file in train_json_files[:5]:
    print(json_file.name)


# JSON이 하나도 없다면 경로가 잘못된 것이므로 실행을 중단합니다.
assert len(train_json_files) > 0, (
    "Training JSON을 찾지 못했습니다. "
    "Google Drive 바로가기와 폴더 경로를 확인하세요."
)

In [ ]:
# JSON 파일을 읽기 위한 도구입니다.
import json


# 첫 번째 Training JSON 파일을 엽니다.
with train_json_files[0].open(
    "r",
    encoding="utf-8-sig",
) as file:

    # JSON 내용을 파이썬 딕셔너리로 변환합니다.
    sample_data = json.load(file)


# JSON에 어떤 항목들이 들어 있는지 확인합니다.
print("전체 JSON 항목:")
print(sample_data.keys())


# 같은 통화를 구분하는 번호를 확인합니다.
print(
    "\nrecordId:",
    sample_data.get("recordId"),
)


# 환자의 증상 정답을 확인합니다.
print(
    "\nsymptom:",
    sample_data.get("symptom"),
)


# 대화가 몇 문장으로 구성되어 있는지 확인합니다.
print(
    "\n발화 개수:",
    len(sample_data.get("utterances", [])),
)


# 첫 번째 발화 내용을 확인합니다.
print(
    "\n첫 번째 발화:",
    sample_data.get("utterances", [])[0],
)

In [ ]:
# 기존 학습 코드에서 데이터 불러오기 함수를 가져옵니다.
from mission3_baseline import load_training_data


# 실제 서울 Training JSON을 모두 불러옵니다.
dataframe = load_training_data(
    TRAIN_LABEL_DIR
)


# 불러온 데이터 개수를 확인합니다.
print(
    "불러온 데이터 개수:",
    len(dataframe),
)


# 표의 처음 5개 데이터를 확인합니다.
display(
    dataframe.head()
)

In [ ]:
# 원래 JSON에 적혀 있던 증상과
# 평가에 사용할 증상을 함께 비교합니다.
display(
    dataframe[
        [
            "label_file_name",
            "original_symptoms",
            "symptoms",
        ]
    ].head(10)
)

In [ ]:
# Path 형태의 경로를 명령어에서 사용할 수 있도록 문자로 바꿉니다.
train_label_path = str(TRAIN_LABEL_DIR)


# 서울 Training JSON을 이용해 모델을 학습합니다.
!python mission3_baseline.py \
  --label_dir "$train_label_path" \
  --output_dir "./artifacts" \
  --threshold 0.5 \
  --test_size 0.2 \
  --seed 42

In [ ]:
# Validation JSON 폴더 경로를 문자로 바꿉니다.
validation_label_path = str(VALID_LABEL_DIR)

# Validation WAV 폴더 경로도 문자로 바꿉니다.
validation_audio_path = str(VALID_AUDIO_DIR)


# 학습된 모델로 Validation 데이터의 증상을 예측합니다.
!python inference.py \
  --audio_dir "$validation_audio_path" \
  --label_dir "$validation_label_path" \
  --ckpt_path "./artifacts/mission3_tfidf_logreg.joblib" \
  --output "./outputs/mission3.csv"